<a href="https://colab.research.google.com/github/nisaa783/scikit-learn-cookbook_Anisa-Khasanah/blob/main/Chapter_9_Text_Processing_and_Multiclass_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Pendahuluan dan Ekstraksi Fitur Teks
Algoritma machine learning bekerja menggunakan representasi numerik, sehingga teks mentah seperti kalimat, ulasan, atau dokumen harus diubah terlebih dahulu menjadi angka melalui proses text preprocessing dan feature extraction. Pendekatan paling dasar adalah Bag-of-Words, yang merepresentasikan setiap dokumen berdasarkan frekuensi kemunculan kata tanpa memperhitungkan urutan tata bahasa secara penuh.

Namun, kata-kata umum yang sering muncul di hampir semua dokumen (seperti kata hubung atau partikel) sering kali kurang informatif untuk membedakan topik. Oleh karena itu, digunakan TF-IDF (Term Frequency-Inverse Document Frequency) untuk memberikan bobot lebih tinggi pada kata yang sering muncul dalam suatu dokumen tertentu, tetapi jarang muncul di keseluruhan koleksi dokumen (corpus).

## 2. Strategi Klasifikasi Multikelas pada Teks
Ketika target prediksi terdiri dari lebih dari dua kategori atau topik, permasalahan berubah menjadi multiclass classification. Pada pemrosesan teks, representasi TF-IDF atau CountVectorizer umumnya digabungkan dengan algoritma klasifikasi seperti Multinomial Naive Bayes atau Logistic Regression di dalam sebuah Pipeline agar proses vektorisasi dan pelatihan berjalan rapi dan terhindar dari data leakage.

**Konsep & Rumus Matematika:**


*   Term Frequency (TF): Mengukur seberapa sering suatu istilah/kata $t$ muncul di dalam dokumen
$d$:$$\text{TF}(t, d) = \frac{\text{jumlah kemunculan } t \text{ di } d}{\text{total kata di } d}$$
*   Inverse Document Frequency (IDF) & TF-IDF: Menekan bobot kata yang terlalu umum di seluruh kumpulan dokumen $D$:$$\text{IDF}(t, D) = \log\left(\frac{1 + \vert{}D\vert{}}{1 + \vert{}\{d \in D : t \in d\}\vert{}}\right) + 1$$$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

## 3. Contoh Program Penerapan 1: Ekstraksi Fitur TF-IDF dan Klasifikasi Teks Multikelas
Contoh program pertama ini menunjukkan bagaimana mengubah teks menjadi matriks fitur TF-IDF dan melatih model MultinomialNB untuk mengklasifikasikan teks ke dalam beberapa kategori.



In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# 1. Membuat dataset teks contoh dengan 3 kategori (0: Olahraga, 1: Teknologi, 2: Kuliner)
texts = [
    "Tim sepak bola mencetak gol kemenangan di menit akhir pertandingan",
    "Pelatih mengganti pemain sayap untuk menambah serangan di babak kedua",
    "Turnamen bulu tangkis dan basket menarik banyak penonton stadion",
    "Model machine learning dan pipeline python mempercepat analisis data",
    "Pengembangan perangkat keras dan komputasi awan terus berkembang pesat",
    "Keamanan jaringan dan analisis paket data penting untuk sistem modern",
    "Resep nasi goreng dan mie kuah hangat sangat cocok disantap malam hari",
    "Kopi hitam dan camilan tradisional disajikan bersama keluarga di rumah",
    "Bahan rempah segar membuat masakan rumahan terasa lebih gurih dan nikmat"
]
labels = [0, 0, 0, 1, 1, 1, 2, 2, 2]
label_names = ["Olahraga", "Teknologi", "Kuliner"]

# 2. Membagi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.33, random_state=42, stratify=labels
)

# 3. Membangun Pipeline TF-IDF + Multinomial Naive Bayes
model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True)),
    ("clf", MultinomialNB(alpha=1.0))
])

# 4. Melatih model dan melakukan prediksi
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Akurasi Klasifikasi Teks: {accuracy_score(y_test, y_pred):.4f}")
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

Akurasi Klasifikasi Teks: 0.3333

Laporan Klasifikasi:
              precision    recall  f1-score   support

    Olahraga       0.00      0.00      0.00         1
   Teknologi       0.33      1.00      0.50         1
     Kuliner       0.00      0.00      0.00         1

    accuracy                           0.33         3
   macro avg       0.11      0.33      0.17         3
weighted avg       0.11      0.33      0.17         3



## 4. Contoh Program Penerapan 2: Menggunakan N-Grams dan Logistic Regression pada Teks

Contoh program kedua ini memanfaatkan rentang n-grams (ngram_range=(1, 2)) agar model tidak hanya melihat kata tunggal (unigram), tetapi juga pasangan kata berurutan (bigram), lalu mengombinasikannya dengan LogisticRegression untuk klasifikasi multikelas.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# 1. Dataset pelatihan sederhana untuk klasifikasi topik
train_texts = [
    "Evaluasi model klasifikasi dan validasi silang pada dataset",
    "Optimasi hyperparameter dan regulasi model regresi linear",
    "Konfigurasi server linux dan aturan firewall jaringan",
    "Pemantauan trafik jaringan dan analisis log keamanan",
    "Perencanaan jadwal latihan rutin dan pemulihan kebugaran",
    "Strategi permainan tim dan latihan fisik harian"
]
train_labels = ["Data & ML", "Data & ML", "Keamanan & Jaringan", "Keamanan & Jaringan", "Kebugaran", "Kebugaran"]

# 2. Pipeline dengan unigram + bigram dan Logistic Regression
text_pipe = Pipeline([
    ("vectorizer", TfidfVectorizer(ngram_range=(1, 2), stop_words="english")),
    ("classifier", LogisticRegression(max_iter=500, random_state=42))
])

text_pipe.fit(train_texts, train_labels)

# 3. Pengujian pada kalimat baru
sample_texts = [
    "Tuning model regresi dan cek performa validasi",
    "Analisis paket jaringan dan konfigurasi server"
]
predictions = text_pipe.predict(sample_texts)
probabilities = text_pipe.predict_proba(sample_texts)

for text, pred, probs in zip(sample_texts, predictions, probabilities):
    print(f"Teks: '{text}'")
    print(f"Prediksi Kategori: {pred}")
    print(f"Probabilitas Kelas: {dict(zip(text_pipe.classes_, probs.round(4)))}\n")

Teks: 'Tuning model regresi dan cek performa validasi'
Prediksi Kategori: Data & ML
Probabilitas Kelas: {np.str_('Data & ML'): np.float64(0.4403), np.str_('Keamanan & Jaringan'): np.float64(0.2799), np.str_('Kebugaran'): np.float64(0.2799)}

Teks: 'Analisis paket jaringan dan konfigurasi server'
Prediksi Kategori: Keamanan & Jaringan
Probabilitas Kelas: {np.str_('Data & ML'): np.float64(0.2677), np.str_('Keamanan & Jaringan'): np.float64(0.4647), np.str_('Kebugaran'): np.float64(0.2675)}

